In [2]:
import pandas as pd

books = pd.read_csv("books_with_categories.csv")

In [3]:
from transformers import pipeline
classifier = pipeline("text-classification",
                      model="j-hartmann/emotion-english-distilroberta-base",
                      top_k = None,
                      device = "cuda")
classifier("I love this!")

Device set to use cuda


[[{'label': 'joy', 'score': 0.9771687984466553},
  {'label': 'surprise', 'score': 0.008528676815330982},
  {'label': 'neutral', 'score': 0.005764591973274946},
  {'label': 'anger', 'score': 0.004419781267642975},
  {'label': 'sadness', 'score': 0.002092393347993493},
  {'label': 'disgust', 'score': 0.001611992483958602},
  {'label': 'fear', 'score': 0.00041385178337804973}]]

In [4]:
import numpy as np

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

def calculate_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotion_labels}
    for prediction in predictions:
        sorted_predictions = sorted(prediction, key=lambda x: x["label"])
        for index, label in enumerate(emotion_labels):
            per_emotion_scores[label].append(sorted_predictions[index]["score"])
    return {label: np.max(scores) for label, scores in per_emotion_scores.items()}

In [5]:
for i in range(10):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [6]:
emotion_scores

{'anger': [np.float64(0.06413373351097107),
  np.float64(0.6126190423965454),
  np.float64(0.06413373351097107),
  np.float64(0.35148340463638306),
  np.float64(0.08141230791807175),
  np.float64(0.23222434520721436),
  np.float64(0.5381830930709839),
  np.float64(0.06413373351097107),
  np.float64(0.3006700873374939),
  np.float64(0.06413373351097107)],
 'disgust': [np.float64(0.27359187602996826),
  np.float64(0.34828463196754456),
  np.float64(0.10400677472352982),
  np.float64(0.15072226524353027),
  np.float64(0.1844952404499054),
  np.float64(0.7271754145622253),
  np.float64(0.15585492551326752),
  np.float64(0.10400677472352982),
  np.float64(0.2794809639453888),
  np.float64(0.17792759835720062)],
 'fear': [np.float64(0.9281678199768066),
  np.float64(0.9425276517868042),
  np.float64(0.9723208546638489),
  np.float64(0.3607073128223419),
  np.float64(0.09504341334104538),
  np.float64(0.051362887024879456),
  np.float64(0.7474286556243896),
  np.float64(0.40449604392051697),


In [7]:
from tqdm import tqdm

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

for i in tqdm(range(len(books))):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

100%|██████████| 5230/5230 [01:52<00:00, 46.69it/s]


In [8]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df["isbn13"] = isbn

In [9]:
books = pd.merge(books, emotions_df, on = "isbn13")

In [10]:
books.to_csv("books_with_emotions.csv", index = False)